# 《One Step Diffusion via Shortcut Models》论文详细解读

**论文：** One Step Diffusion via Shortcut Models  
**作者：** Kevin Frans、Danijar Hafner、Sergey Levine、Pieter Abbeel  
**会议：** ICLR 2025  
**论文：** https://arxiv.org/abs/2410.12557  
**开源代码：** https://github.com/kvfrans/shortcut-models

我阅读了论文正文第 1～6 节，以及附录 A、B 的实验结果和训练细节。下面按照论文的实际结构展开，并将核心公式重新推导。

我们继续统一使用：

$$
\boxed{
x_0=\text{初始噪声},\qquad
x_1=\text{真实数据},\qquad
0\rightarrow1.
}
$$

这恰好也是 Shortcut 原论文采用的时间方向，不需要再进行时间反演。

全文重点放在三个问题上：

1. 为什么普通 Flow Matching 即使训练到了最优，也无法直接一步生成多模态数据？
2. Shortcut 如何利用两个小步长预测构造一个大步长的训练目标？
3. 为什么这种自举训练能够从零开始，并且不需要预训练教师模型？

---

# 第一部分：全文核心结论

## 1. Shortcut Models 究竟提出了什么？

Shortcut Models 的核心贡献不是提出新的神经网络结构，而是提出一种新的**速度场参数化与训练方式**。

普通 Flow Matching 学习：

$$
\boxed{
v_\theta(x_t,t)
}
\tag{1.1}
$$

即当前状态 $x_t$、当前时间 $t$ 下的瞬时速度。

Shortcut 将时间步长 $d$ 也作为网络输入：

$$
\boxed{
s_\theta(x_t,t,d).
}
\tag{1.2}
$$

其中：

- $x_t\in\mathbb R^D$：当前状态；
- $t\in[0,1]$：当前时间；
- $d\in[0,1-t]$：希望向前跨越的时间长度；
- $s_\theta\in\mathbb R^D$：预测的区间平均速度；
- $\theta$：网络参数。

采样公式为：

$$
\boxed{
x_{t+d}
=
x_t+d\,s_\theta(x_t,t,d).
}
\tag{1.3}
$$

如果：

$$
t=0,\qquad d=1,
$$

则：

$$
\boxed{
\hat x_1=x_0+s_\theta(x_0,0,1).
}
\tag{1.4}
$$

一次网络推理即可生成数据。

但关键问题是：

**训练时，我们并不知道真实的 $s(x_t,t,d)$，那么怎么监督网络？**

论文提出两类训练信号。

### 第一类：瞬时速度监督

当：

$$
d=0,
$$

Shortcut 退化为普通 Flow Matching：

$$
\boxed{
s_\theta(x_t,t,0)\approx v(x_t,t).
}
\tag{1.5}
$$

可以使用已知的条件速度：

$$
V=x_1-x_0
$$

进行训练。

### 第二类：大步长自举监督

对于：

$$
d>0,
$$

要求：

$$
\boxed{
\begin{aligned}
s(x_t,t,2d)
={}&
\frac12s(x_t,t,d)\\
&+
\frac12s(x_{t+d},t+d,d).
\end{aligned}
}
\tag{1.6}
$$

其中：

$$
x_{t+d}
=
x_t+d\,s(x_t,t,d).
$$

这个公式表达：

> 从当前状态直接跨越 $2d$ 的平均速度，应该等于连续跨越两个 $d$ 区间的平均速度的平均值。

训练时，用当前模型的两个小步长预测构造大步长 Target，再对 Target 停止梯度。

这是一种自举训练。

---

## 2. Shortcut 最重要的四个结论

**结论一：普通 FM 的瞬时速度不适合直接用于大步长生成。**

尤其在多模态数据分布中，一步 Euler 会把样本推向条件均值。

**结论二：Shortcut 学习的是有限时间区间的平均速度。**

$$
s(x_t,t,d)
=
\frac{x_{t+d}-x_t}{d}.
$$

这里的终点必须来自边缘 ODE，而不是随机配对样本的线性插值。

**结论三：不需要完整积分得到大步长的监督标签。**

通过：

$$
\text{两个小步长}\rightarrow\text{一个大步长}
$$

构造自监督 Target。

**结论四：可以在同一次训练中联合学习所有步长。**

不需要先训练一个 FM 教师，再进行单独蒸馏。

论文使用 $M=128$ 的时间分辨率，同时训练不同的 Shortcut 长度，并支持 1、4、128 步等不同推理预算。作者估算训练计算量比基础扩散模型增加约 16%。[arXiv](https://arxiv.org/html/2410.12557?utm_source=chatgpt.com)

---

# 第二部分：Background——为什么普通 Flow Matching 一步生成会失败？

这是论文第 2 节，也是理解 Shortcut 必须掌握的部分。

## 3. Flow Matching 的训练目标

定义：

$$
x_0\sim p_0=\mathcal N(0,I),
$$

$$
x_1\sim p_1=p_{\mathrm{data}}.
$$

采用线性插值：

$$
\boxed{
x_t=(1-t)x_0+tx_1.
}
\tag{3.1}
$$

对时间求导：

$$
\frac{dx_t}{dt}
=
-x_0+x_1.
$$

得到条件速度：

$$
\boxed{
V=x_1-x_0.
}
\tag{3.2}
$$

注意这里的 $V$ 对于固定样本对 $(x_0,x_1)$ 是常量。

但给定中间状态 $x_t$，可能存在许多不同的样本对经过这个位置。

因此：

$$
V\mid x_t
$$

是随机变量。

最终 Flow Matching 学习的是条件期望：

$$
\boxed{
v(x,t)=\mathbb E[V\mid x_t=x].
}
\tag{3.3}
$$

训练损失：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E
\left[
\|v_\theta(x_t,t)-V\|^2
\right].
}
\tag{3.4}
$$

由于平方误差的最优预测是条件期望：

$$
\boxed{
v_\theta^*(x,t)=v(x,t).
}
\tag{3.5}
$$

这就是标准 Flow Matching。

---

## 4. 为什么最优 FM 仍然不能一步生成？

这一点可以严格推导，而不只是说 Euler 积分误差较大。

假设：

$$
x_0
$$

与：

$$
x_1
$$

独立采样。

这是论文采用的随机配对方式。

在：

$$
t=0,
$$

有：

$$
x_t=x_0.
$$

因此：

$$
v(x_0,0)
=
\mathbb E[x_1-x_0\mid x_0].
$$

由于：

$$
x_1\perp x_0,
$$

所以：

$$
\mathbb E[x_1\mid x_0]
=
\mathbb E[x_1].
$$

因此：

$$
\begin{aligned}
v(x_0,0)
&=
\mathbb E[x_1\mid x_0]-x_0\\
&=
\mathbb E[x_1]-x_0.
\end{aligned}
$$

定义数据均值：

$$
\mu_{\mathrm{data}}=\mathbb E[x_1].
$$

得到：

$$
\boxed{
v(x_0,0)=\mu_{\mathrm{data}}-x_0.
}
\tag{4.1}
$$

现在直接使用一步 Euler：

$$
\hat x_1=x_0+v(x_0,0).
$$

代入：

$$
\begin{aligned}
\hat x_1
&=
x_0+\mu_{\mathrm{data}}-x_0\\
&=
\mu_{\mathrm{data}}.
\end{aligned}
$$

所以：

$$
\boxed{
\hat x_1=\mathbb E[x_1].
}
\tag{4.2}
$$

**无论初始噪声是什么，最优 FM 的一步 Euler 都输出数据均值。**

这意味着，对于多模态数据：

$$
p_{\mathrm{data}}
=
\frac12\mathcal N(-3,\sigma^2)
+
\frac12\mathcal N(3,\sigma^2),
$$

其均值为：

$$
\mu_{\mathrm{data}}=0.
$$

那么一步 FM 的输出：

$$
\hat x_1=0.
$$

但真实数据主要集中在：

$$
-3,\quad3.
$$

所以模型会生成两个模式中间的位置。

这就是论文所说的 **Few-step Ambiguity**。

需要强调，这个结论成立于随机独立配对、精确学习边缘速度、直接使用初始速度做完整一步 Euler 的设定。它不是说 FM 的完整 ODE 不能生成多模态数据。

实际上，沿着 ODE 逐步积分：

$$
\frac{dx_t}{dt}=v(x_t,t)
$$

仍然可以把初始噪声分布输运到目标数据分布。

问题出在：

$$
\boxed{
\text{用初始瞬时速度代替完整区间的积分。}
}
$$

这也是 Shortcut 的直接动机。[arXiv](https://arxiv.org/html/2410.12557?utm_source=chatgpt.com)

---

# 第三部分：Shortcut Models 的数学定义

对应论文第 3 节。

## 5. 从瞬时速度扩展到有限时间平均速度

考虑真实边缘 ODE：

$$
\boxed{
\frac{dx_\tau}{d\tau}=v(x_\tau,\tau).
}
\tag{5.1}
$$

从当前状态：

$$
x_t
$$

出发，积分到：

$$
t+d.
$$

有：

$$
x_{t+d}
=
x_t+\int_t^{t+d}v(x_\tau,\tau)\,d\tau.
$$

定义：

$$
\boxed{
s(x_t,t,d)
=
\frac1d
\int_t^{t+d}
v(x_\tau,\tau)\,d\tau.
}
\tag{5.2}
$$

因此：

$$
\boxed{
x_{t+d}
=
x_t+d\,s(x_t,t,d).
}
\tag{5.3}
$$

这就是 Shortcut 的数学定义。

注意，$s$ 在这篇论文里表示 **Shortcut velocity**，不是我们前面 MeanFlow 笔记中的起始时间变量。

为了保持论文符号，下面继续使用：

$$
s_\theta(x_t,t,d).
$$

## 6. 为什么 $d\rightarrow0$ 时退化为瞬时速度？

根据：

$$
s(x_t,t,d)
=
\frac1d
\int_t^{t+d}v(x_\tau,\tau)\,d\tau,
$$

如果速度场连续，那么：

$$
\lim_{d\rightarrow0^+}
\frac1d
\int_t^{t+d}v(x_\tau,\tau)\,d\tau
=
v(x_t,t).
$$

因此：

$$
\boxed{
s(x_t,t,0)=v(x_t,t).
}
\tag{6.1}
$$

这里 $d=0$ 是通过连续延拓定义的。

所以 Shortcut 可以理解为：

$$
\boxed{
\begin{aligned}
d=0 &: \text{瞬时速度};\\
d>0 &: \text{有限时间平均速度};\\
d=1,t=0 &: \text{完整生成区间的平均速度}.
\end{aligned}
}
\tag{6.2}
$$

这使得同一个网络能够覆盖不同推理步数。

---

## 7. 核心公式：一个整步等于两个半步

这是全文最重要的推导。

考虑：

$$
x_t\rightarrow x_{t+d}\rightarrow x_{t+2d}.
$$

### 7.1 直接跨越 $2d$

根据 Shortcut 定义：

$$
\boxed{
x_{t+2d}
=
x_t+2d\,s(x_t,t,2d).
}
\tag{7.1}
$$

### 7.2 先跨越第一个 $d$

$$
\boxed{
x_{t+d}
=
x_t+d\,s(x_t,t,d).
}
\tag{7.2}
$$

### 7.3 再跨越第二个 $d$

从：

$$
x_{t+d}
$$

出发：

$$
\boxed{
x_{t+2d}
=
x_{t+d}
+
d\,s(x_{t+d},t+d,d).
}
\tag{7.3}
$$

将式 (7.2) 代入：

$$
\begin{aligned}
x_{t+2d}
={}&
x_t+d\,s(x_t,t,d)\\
&+
d\,s(x_{t+d},t+d,d).
\end{aligned}
\tag{7.4}
$$

### 7.4 两种方式的终点必须相同

由式 (7.1) 和式 (7.4)：

$$
\begin{aligned}
x_t+2d\,s(x_t,t,2d)
={}&
x_t+d\,s(x_t,t,d)\\
&+
d\,s(x_{t+d},t+d,d).
\end{aligned}
$$

两边消去：

$$
x_t.
$$

得到：

$$
\begin{aligned}
2d\,s(x_t,t,2d)
={}&
d\,s(x_t,t,d)\\
&+
d\,s(x_{t+d},t+d,d).
\end{aligned}
$$

两边除以：

$$
2d.
$$

最终：

$$
\boxed{
\begin{aligned}
s(x_t,t,2d)
={}&
\frac12s(x_t,t,d)\\
&+
\frac12s(x_{t+d},t+d,d).
\end{aligned}
}
\tag{7.5}
$$

这就是论文公式 (4)。

### 7.5 为什么两个速度取平均？

因为它们对应的时间长度相同。

完整区间长度：

$$
2d.
$$

两个子区间分别长：

$$
d.
$$

总位移：

$$
\Delta x
=
d\,s_1+d\,s_2.
$$

完整区间平均速度：

$$
s_{\mathrm{large}}
=
\frac{\Delta x}{2d}.
$$

因此：

$$
\boxed{
s_{\mathrm{large}}=\frac{s_1+s_2}{2}.
}
\tag{7.6}
$$

如果两个区间长度不同，例如：

$$
d_1,\quad d_2,
$$

那么一般形式为：

$$
\boxed{
s_{\mathrm{large}}
=
\frac{
d_1s_1+d_2s_2
}{
d_1+d_2
}.
}
\tag{7.7}
$$

论文选择两个相等的半步，是为了构造简单的二叉递归训练过程。

---

## 8. 一个重要细节：第二个半步的输入从哪里来？

论文特别强调：

**第二个半步不能直接使用原始样本对的线性插值点。**

假设训练样本为：

$$
x_0,x_1.
$$

我们可以构造：

$$
x_t^{\mathrm{interp}}
=
(1-t)x_0+tx_1.
$$

但第二个半步需要的状态应该是：

$$
\boxed{
\hat x_{t+d}
=
x_t+d\,s_\theta(x_t,t,d).
}
\tag{8.1}
$$

而不是：

$$
(1-t-d)x_0+(t+d)x_1.
$$

为什么？

因为原始线性插值：

$$
(1-\tau)x_0+\tau x_1
$$

是一对随机配对样本的条件路径。

而 Shortcut 希望学习的是边缘 ODE 的流映射。

这两个路径一般不同。

如果第二步直接使用线性插值点，就会重新引入随机配对路径的歧义。

因此，论文使用模型自身预测的第一步终点，作为第二步的输入。

这使得训练约束真正作用于模型所学习的 ODE 轨迹。[arXiv](https://arxiv.org/html/2410.12557?utm_source=chatgpt.com)

---

# 第四部分：Shortcut 的完整训练方法

## 9. 第一部分 Loss：Flow Matching

首先，从训练数据采样：

$$
x_0\sim\mathcal N(0,I),
\qquad
x_1\sim p_{\mathrm{data}}.
$$

随机采样：

$$
t\sim\mathcal U(0,1).
$$

构造：

$$
x_t=(1-t)x_0+tx_1.
$$

条件速度：

$$
V=x_1-x_0.
$$

让网络在：

$$
d=0
$$

时预测：

$$
s_\theta(x_t,t,0).
$$

损失：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E
\left[
\|s_\theta(x_t,t,0)-V\|^2
\right].
}
\tag{9.1}
$$

其最优预测：

$$
\boxed{
s_\theta^*(x,t,0)
=
\mathbb E[V\mid x_t=x]
=
v(x,t).
}
\tag{9.2}
$$

这一项负责学习正确的瞬时速度场。

**它是整个 Shortcut 自举过程的基础。**

如果没有这个监督，只使用两个半步等于一个整步的约束，那么网络可能学到一些自洽但错误的映射。

例如：

$$
s_\theta(x,t,d)=0
$$

对所有输入都成立。

那么：

$$
s_\theta(x,t,2d)=0,
$$

两个半步的平均值也为零。

因此自一致性损失为零。

但显然，这个模型不能把高斯噪声变换成真实数据。

所以：

$$
\boxed{
\text{自一致性约束本身不能决定正确的生成分布。}
}
$$

必须通过：

$$
\mathcal L_{\mathrm{FM}}
$$

将模型锚定到真实数据对应的边缘速度场。

---

## 10. 第二部分 Loss：Self-Consistency

现在考虑：

$$
d>0.
$$

我们希望学习：

$$
s_\theta(x_t,t,2d).
$$

但不知道真实标签：

$$
s^*(x_t,t,2d).
$$

论文使用当前模型生成两个小步长预测。

### 10.1 第一次网络预测

输入：

$$
(x_t,t,d).
$$

输出：

$$
\boxed{
s_1=s_\theta(x_t,t,d).
}
\tag{10.1}
$$

### 10.2 计算第一个小步长的终点

$$
\boxed{
\hat x_{t+d}=x_t+d\,s_1.
}
\tag{10.2}
$$

注意这里的：

$$
\hat x_{t+d}
$$

是模型预测得到的，不是训练数据直接提供的。

### 10.3 第二次网络预测

输入：

$$
(\hat x_{t+d},t+d,d).
$$

输出：

$$
\boxed{
s_2=s_\theta(\hat x_{t+d},t+d,d).
}
\tag{10.3}
$$

### 10.4 构造大步长 Target

根据前面推导的自一致性公式：

$$
\boxed{
s_{\mathrm{target}}
=
\frac{s_1+s_2}{2}.
}
\tag{10.4}
$$

代入网络表达式：

$$
\boxed{
\begin{aligned}
s_{\mathrm{target}}
=
\frac12\Big[
&s_\theta(x_t,t,d)\\
+&s_\theta(
x_t+d\,s_\theta(x_t,t,d),
t+d,d)
\Big].
\end{aligned}
}
\tag{10.5}
$$

这就是 Shortcut 最核心的训练目标。

注意它与 MeanFlow 一样，也使用当前模型构造 Target。

但这里不需要 JVP，而是直接执行两次小步长网络预测。

### 10.5 停止梯度

定义：

$$
\boxed{
\bar s_{\mathrm{target}}
=
\operatorname{sg}(s_{\mathrm{target}}).
}
\tag{10.6}
$$

然后执行第三次网络预测：

$$
\boxed{
s_{\mathrm{pred}}
=
s_\theta(x_t,t,2d).
}
\tag{10.7}
$$

计算：

$$
\boxed{
\mathcal L_{\mathrm{SC}}
=
\mathbb E
\left[
\|s_{\mathrm{pred}}-\bar s_{\mathrm{target}}\|^2
\right].
}
\tag{10.8}
$$

在这一次反向传播中：

- $s_1$：只用于构造 Target；
- $s_2$：只用于构造 Target；
- $\hat x_{t+d}$：只用于构造 Target；
- $s_{\mathrm{pred}}$：参与梯度更新。

因此，Target 分支不会反向传播梯度。

---

## 11. 用一个数值例子理解整个 Target

假设一维状态：

$$
x_t=1.
$$

当前时间：

$$
t=0.2.
$$

希望学习大步长：

$$
2d=0.4.
$$

因此：

$$
d=0.2.
$$

### 第一步：预测第一个半步

假设网络输出：

$$
s_1=s_\theta(1,0.2,0.2)=3.
$$

于是：

$$
\hat x_{0.4}
=
1+0.2\times3
=
1.6.
$$

### 第二步：预测第二个半步

网络输入：

$$
(1.6,0.4,0.2).
$$

假设输出：

$$
s_2=5.
$$

于是：

$$
\hat x_{0.6}
=
1.6+0.2\times5
=
2.6.
$$

两个半步总位移：

$$
2.6-1=1.6.
$$

完整时间长度：

$$
0.4.
$$

因此平均速度：

$$
s_{\mathrm{target}}
=
\frac{1.6}{0.4}
=
4.
$$

也可以直接计算：

$$
s_{\mathrm{target}}
=
\frac{3+5}{2}
=
4.
$$

所以大步长网络应该预测：

$$
\boxed{
s_\theta(1,0.2,0.4)\approx4.
}
$$

假设当前网络输出：

$$
s_{\mathrm{pred}}=2.5.
$$

那么：

$$
\mathcal L_{\mathrm{SC}}
=
(2.5-4)^2
=
2.25.
$$

网络会更新参数，使：

$$
s_\theta(1,0.2,0.4)
$$

向 4 靠近。

下一次迭代，模型参数更新，两个半步预测也可能变化，于是 Target 会重新计算。

所以这是一个动态的自举过程。

---

## 12. 为什么两个半步能够逐渐训练出一步生成？

论文采用二进制时间尺度。

设最小时间单位：

$$
h=\frac1{128}.
$$

训练的步长包括：

$$
\boxed{
d\in
\left\{
0,
\frac1{128},
\frac1{64},
\frac1{32},
\frac1{16},
\frac18,
\frac14,
\frac12,
1
\right\}.
}
\tag{12.1}
$$

这里 $d=0$ 是瞬时速度监督分支；其余为有限时间 Shortcut。

训练逻辑可以理解为：

$$
\boxed{
\begin{aligned}
d=0
&\xrightarrow{\mathrm{FM}}
v(x,t)\\
&\Downarrow\\
d=\frac1{128}
&\xrightarrow{\mathrm{bootstrap}}
\text{极短区间速度}\\
&\Downarrow\\
d=\frac1{64}
&\xrightarrow{\mathrm{bootstrap}}
\text{较短区间速度}\\
&\Downarrow\\
d=\frac1{32}
&\Downarrow\\
&\vdots\\
d=\frac12
&\Downarrow\\
d=1
&\rightarrow
\text{完整生成区间平均速度}.
\end{aligned}
}
\tag{12.2}
$$

例如，学习：

$$
d=1
$$

时，Target 来自两个：

$$
d=\frac12
$$

的预测。

学习：

$$
d=\frac12
$$

时，Target 来自两个：

$$
d=\frac14
$$

的预测。

如此递归。

但是需要特别强调：

**这并不意味着论文先训练完 $d=1/128$，再训练 $d=1/64$，最后训练 $d=1$。**

论文没有采用这种阶段式训练。

所有步长在同一次训练过程中随机采样、联合优化。

上述递归关系描述的是监督信息的数学依赖，而不是训练阶段的先后顺序。

这是 Shortcut 与传统逐阶段蒸馏的重要区别。

---

## 13. 最小步长的特殊处理

这里存在一个容易忽略的问题。

假设要训练：

$$
d=\frac1{128}.
$$

按照一般规则，需要两个：

$$
d=\frac1{256}
$$

的预测。

但论文只定义了最小非零步长：

$$
\frac1{128}.
$$

怎么办？

论文的做法是：

**最小非零步长使用 $d=0$ 的瞬时速度预测构造 Target。**

例如，令：

$$
h=\frac1{128}.
$$

从：

$$
x_t
$$

出发，执行两个：

$$
\frac h2
$$

长度的 Euler 更新，但两次网络都查询：

$$
d=0
$$

的瞬时速度分支。

第一步：

$$
\boxed{
\hat x_{t+h/2}
=
x_t+\frac h2s_\theta(x_t,t,0).
}
\tag{13.1}
$$

第二步：

$$
\boxed{
\hat x_{t+h}
=
\hat x_{t+h/2}
+
\frac h2
s_\theta(\hat x_{t+h/2},t+h/2,0).
}
\tag{13.2}
$$

因此：

$$
\boxed{
\begin{aligned}
s_{\mathrm{target}}(x_t,t,h)
=
\frac12\Big[
&s_\theta(x_t,t,0)\\
+&s_\theta(\hat x_{t+h/2},t+h/2,0)
\Big].
\end{aligned}
}
\tag{13.3}
$$

这样，最小非零 Shortcut 就有了来自瞬时速度场的监督。

之后再通过二叉自举向更大的时间区间传递。

---

## 14. 为什么这种训练不需要教师模型？

传统蒸馏通常有：

$$
\theta_{\mathrm{teacher}}
$$

和：

$$
\theta_{\mathrm{student}}.
$$

教师先训练完成，然后提供监督。

Shortcut 只有一套需要训练的主模型：

$$
\theta.
$$

但为了提高稳定性，论文实际还维护了 EMA 参数：

$$
\theta_{\mathrm{EMA}}.
$$

其更新方式为：

$$
\boxed{
\theta_{\mathrm{EMA}}
\leftarrow
\beta\theta_{\mathrm{EMA}}
+
(1-\beta)\theta.
}
\tag{14.1}
$$

论文使用：

$$
\boxed{
\beta=0.999.
}
$$

EMA 模型用于生成自一致性 Target，并用于最终评价。

因此，更准确的说法是：

> Shortcut 不需要预训练好的独立教师模型，而是在同一次训练中使用当前模型的 EMA 副本提供较稳定的自举监督。

这与“完全不使用任何教师形式的网络副本”不是一回事。

---

# 第五部分：训练细节与稳定性设计

对应论文第 3.1 节。

## 15. 为什么 FM 样本要占 75%？

论文采用：

$$
75\%
$$

的经验速度监督，以及：

$$
25\%
$$

的自一致性监督。

即：

$$
\boxed{
P(\mathrm{FM})=0.75,
\qquad
P(\mathrm{SC})=0.25.
}
\tag{15.1}
$$

这里有两个原因。

### 原因一：两种 Target 的统计性质不同

FM Target：

$$
V=x_1-x_0
$$

来自随机配对。

在给定 $x_t$ 时，可能存在很多不同的 $V$，因此具有较大的条件方差。

Shortcut Target：

$$
s_{\mathrm{target}}
=
\frac{s_1+s_2}{2}
$$

来自模型的确定性预测。

给定输入和模型参数后，Target 基本是确定的。

因此，两种监督信号的随机性不同。

### 原因二：自一致性 Target 更昂贵

FM 样本通常只需要一次主网络预测。

自一致性样本需要：

1. 第一个半步预测；
2. 第二个半步预测；
3. 一个整步预测。

其中两个半步预测不需要反向传播。

如果所有样本都使用自一致性监督，训练计算成本会明显增加。

所以论文选择只对 25% 的 batch 构造自举 Target。

---

## 16. 训练计算量为什么只增加约 16%？

论文使用一个简化的计算成本模型。

假设：

$$
\text{一次前向传播成本}=1.
$$

一次反向传播约为：

$$
2.
$$

普通 FM 训练一个样本：

$$
C_{\mathrm{FM}}=1+2=3.
$$

Shortcut 的主预测分支仍然需要：

$$
1+2=3.
$$

但其中 25% 的样本还需要两次额外的前向传播来构造 Target。

因此：

$$
C_{\mathrm{Shortcut}}
=
3+0.25\times2.
$$

得到：

$$
C_{\mathrm{Shortcut}}=3.5.
$$

相对增加：

$$
\frac{3.5-3}{3}
=
\frac16
\approx16.7\%.
$$

因此：

$$
\boxed{
\text{额外训练计算量}\approx16\%.
}
\tag{16.1}
$$

这是论文采用的近似 FLOPs 估计，不是所有硬件上的实际运行时间都只增加 16%。

---

## 17. Weight Decay 为什么重要？

Shortcut 的 Target 来自当前模型。

在训练初期：

$$
s_\theta(x,t,d)
$$

可能非常不准确。

因此，两个小步长产生的 Target 也可能不准确。

如果网络立即大幅拟合这些错误 Target，就可能产生自举误差放大。

论文发现，较大的 weight decay 对训练稳定性非常重要。

使用：

$$
\boxed{
\lambda_{\mathrm{wd}}=0.1.
}
\tag{17.1}
$$

它有助于限制参数过度增长，并改善训练初期的自举稳定性。

论文还发现，使用 EMA 参数构造 Target 可以减少训练过程中的预测震荡。

这里的 EMA 和 weight decay 都属于训练稳定性设计，而不是 Shortcut 自一致性公式本身的数学要求。[arXiv](https://arxiv.org/html/2410.12557?utm_source=chatgpt.com)

---

## 18. 为什么时间采样要离散化？

论文不是任意连续采样所有：

$$
(t,d).
$$

而是选择与推理时实际使用的时间网格相对应的点。

例如：

$$
d=\frac14.
$$

如果希望进行四步生成，网络会被查询：

$$
t=0,\frac14,\frac12,\frac34.
$$

所以训练时主要关注这些位置。

对于：

$$
d=\frac12,
$$

则查询：

$$
t=0,\frac12.
$$

对于：

$$
d=1,
$$

只有：

$$
t=0.
$$

因此：

$$
\boxed{
t\in\{0,d,2d,\ldots,1-d\}.
}
\tag{18.1}
$$

这种设计让网络重点学习真正会在推理中出现的输入组合。

同时也保证：

$$
t+d\leq1.
$$

---

## 19. CFG 如何处理？

对于类别条件生成，定义：

$$
v_{\mathrm{cond}}(x,t\mid c)
$$

和：

$$
v_{\mathrm{uncond}}(x,t).
$$

CFG 速度：

$$
\boxed{
v_{\mathrm{CFG}}
=
\omega v_{\mathrm{cond}}
+
(1-\omega)v_{\mathrm{uncond}}.
}
\tag{19.1}
$$

论文发现：

- 在小步长下，CFG 有帮助；
- 在大步长下，直接线性组合条件和无条件的 Shortcut 预测容易引入误差。

因此，论文采用的策略是：

**在瞬时速度分支使用 CFG，并通过训练过程把这种引导效果传递到较大步长的 Shortcut。**

论文 ImageNet 实验使用 CFG scale：

$$
\boxed{
\omega=1.5.
}
$$

最终生成时不需要再次进行条件、无条件两次推理。

这里还有一个限制：CFG 强度需要在训练前确定，不能像普通扩散模型那样在推理时自由调整所有强度。[arXiv](https://arxiv.org/html/2410.12557?utm_source=chatgpt.com)

---

# 第六部分：实验结果详解

对应论文第 5 节及附录 A、B。

## 20. 实验设置

论文主要使用：

- CelebA-HQ 256×256：无条件人脸生成；
- ImageNet 256×256：类别条件图像生成；
- Push-T、Transport：机器人控制任务。

图像生成模型采用 DiT，并在预训练 VAE 的潜在空间中训练。

主要配置：

| 参数 | 配置 |
|---|---|
| 基础模型 | DiT-B |
| 图像分辨率 | 256×256 |
| 潜空间大小 | $32\times32\times4$ |
| Optimizer | AdamW |
| Learning rate | $10^{-4}$ |
| Weight decay | 0.1 |
| EMA decay | 0.999 |
| FM 样本比例 | 75% |
| 自一致性样本比例 | 25% |
| 基础时间分辨率 | 128 |
| CelebA-HQ batch size | 64 |
| ImageNet batch size | 256 |
| CelebA-HQ 训练步数 | 400K |
| ImageNet 训练步数 | 800K |

这些是论文附录 B 报告的实际配置。[arXiv](https://arxiv.org/html/2410.12557?utm_source=chatgpt.com)

---

## 21. 实验一：与其他一步生成方法比较

论文使用相同的 DiT-B 架构和相近的计算预算，对比不同训练方法。

### 21.1 CelebA-HQ 256×256

FID 越小越好。

| 方法 | 128 步 | 4 步 | 1 步 |
|---|---:|---:|---:|
| Diffusion | 23.0 | 123.4 | 132.2 |
| Flow Matching | 7.3 | 63.3 | 280.5 |
| Consistency Training | 53.7 | 19.0 | 33.2 |
| Reflow | 16.1 | 18.4 | 23.2 |
| Progressive Distillation | 302.9 | 251.3 | **14.8** |
| **Shortcut** | **6.9** | **13.8** | **20.5** |

需要注意，部分方法并非为所有采样步数设计，论文使用括号标记了这些非目标配置；这里为了阅读方便省略了括号。

Shortcut 在 1 步时取得：

$$
\boxed{
\mathrm{FID}=20.5.
}
$$

普通 FM 一步：

$$
\mathrm{FID}=280.5.
$$

差距非常明显。

但 Progressive Distillation 的一步 FID 为：

$$
14.8,
$$

比 Shortcut 更低。

因此，论文的优势并不是所有情况下的一步 FID 都最好，而是**无需独立蒸馏阶段，同时支持不同推理步数**。

### 21.2 ImageNet 256×256

| 方法 | 128 步 | 4 步 | 1 步 |
|---|---:|---:|---:|
| Diffusion | 39.7 | 464.5 | 467.2 |
| Flow Matching | 17.3 | 108.2 | 324.8 |
| Consistency Training | 42.8 | 43.0 | 69.7 |
| Reflow | 16.9 | 32.8 | 44.8 |
| Progressive Distillation | 201.9 | 142.5 | **35.6** |
| **Shortcut** | **15.5** | **28.3** | **40.3** |

Shortcut 一步：

$$
\boxed{
\mathrm{FID}=40.3.
}
$$

普通 FM 一步：

$$
\mathrm{FID}=324.8.
$$

说明 Shortcut 在低 NFE 情况下明显改善了生成质量。

同时，Shortcut 的 128 步 FID 为：

$$
15.5,
$$

与普通 FM 的：

$$
17.3
$$

接近，甚至略好。

作者提出一种可能解释：自一致性约束可能具有隐式正则化作用。不过论文并没有将这一解释作为已经证明的结论。[arXiv](https://arxiv.org/html/2410.12557?utm_source=chatgpt.com)

---

## 22. 实验二：不同采样步数的表现

论文比较不同 NFE 下的生成效果。

对于普通 FM：

$$
x_{t+d}\approx x_t+d\,v_\theta(x_t,t).
$$

当：

$$
d
$$

增大时，积分误差可能显著增加。

Shortcut 则直接训练：

$$
s_\theta(x_t,t,d).
$$

因此，在较大的时间跨度下仍然可以预测相对准确的终点。

实验观察到：

- 普通 FM 在少步数下容易出现模糊和模式坍缩；
- Shortcut 的少步生成通常能保持整体图像结构；
- Shortcut 的误差更多体现在局部细节；
- 同一初始噪声可以使用不同 NFE 重新生成，增加步数通常能够改善质量。

这意味着 Shortcut 具有一个实用特性：

**先使用一步快速生成，如果需要更高质量，可以使用同一噪声进行多步生成。**

---

## 23. 实验三：模型规模扩展

论文进一步扩大模型规模。

对于 DiT-XL，ImageNet-256 的结果为：

| 模型 | NFE | FID ↓ |
|---|---:|---:|
| Shortcut-XL | 128 | 3.8 |
| Shortcut-XL | 4 | 7.8 |
| Shortcut-XL | 1 | 10.6 |

其中：

$$
\boxed{
\mathrm{FID}_{1\text{-step}}=10.6.
}
$$

这比 DiT-B 的一步结果明显改善。

作者希望验证：

**依赖自举 Target 的模型，在规模扩大后是否仍然能够持续受益？**

实验结果表明，至少在论文测试的规模范围内，Shortcut 的生成质量会随着模型规模增加而改善。

但这属于经验观察，并不意味着所有自举训练方法都具有相同的扩展规律。[arXiv](https://arxiv.org/html/2410.12557?utm_source=chatgpt.com)

---

## 24. 实验四：噪声潜空间插值

论文还研究了初始噪声之间的插值。

采样两个噪声：

$$
x_0^{(a)},x_0^{(b)}
\sim\mathcal N(0,I).
$$

定义：

$$
\boxed{
x_0(\alpha)
=
\alpha x_0^{(a)}
+
\sqrt{1-\alpha^2}\,x_0^{(b)}.
}
\tag{24.1}
$$

其中：

$$
\alpha\in[0,1].
$$

由于两个噪声独立，且都是标准高斯：

$$
\operatorname{Cov}(x_0(\alpha))
=
\alpha^2I+(1-\alpha^2)I
=
I.
$$

所以：

$$
\boxed{
x_0(\alpha)\sim\mathcal N(0,I).
}
\tag{24.2}
$$

然后进行一步生成：

$$
\boxed{
\hat x_1(\alpha)
=
x_0(\alpha)
+
s_\theta(x_0(\alpha),0,1).
}
\tag{24.3}
$$

论文观察到，随着 $\alpha$ 连续变化，生成图像也会呈现相对平滑的语义变化。

这说明模型学习的噪声到数据映射具有一定的连续结构。

不过，插值平滑本身不能直接证明模型生成分布准确。

---

## 25. 实验五：机器人策略生成

论文还将 Shortcut 用于 Diffusion Policy。

传统 Diffusion Policy 需要多次去噪才能生成动作。

Shortcut 希望：

$$
\boxed{
a_1=a_0+s_\theta(a_0,0,1\mid o).
}
\tag{25.1}
$$

其中：

- $a_0$：动作空间中的初始噪声；
- $a_1$：生成动作或动作序列；
- $o$：环境观测条件。

结果：

| 方法 | NFE | Push-T | Transport |
|---|---:|---:|---:|
| Diffusion Policy | 100 | 0.95 | 1.00 |
| Diffusion Policy | 1 | 0.12 | 0.00 |
| LSTM-GMM | 1 | 0.67 | 0.76 |
| BET | 1 | 0.79 | 0.38 |
| **Shortcut Policy** | **1** | **0.87** | **0.80** |

可以看到：

普通 Diffusion Policy 直接压缩为一步，性能严重下降。

Shortcut Policy 在一步生成下仍然保持较好的任务表现。

但它尚未完全达到 100 步 Diffusion Policy 的性能。

这个实验说明 Shortcut 的方法不局限于图像，也能够用于条件动作生成。[arXiv](https://arxiv.org/html/2410.12557?utm_source=chatgpt.com)

---

# 第七部分：Discussion——论文的局限性

论文第 6 节讨论了两个重要限制。

## 26. 一步生成与多步生成仍然存在差距

即使使用 Shortcut：

$$
\mathrm{FID}_{1\text{-step}}
$$

通常仍然明显高于：

$$
\mathrm{FID}_{128\text{-step}}.
$$

例如 DiT-XL：

$$
10.6\quad\text{vs.}\quad3.8.
$$

说明 Shortcut 并没有完全解决一步生成的精度问题。

它只是显著缩小了低 NFE 与多步采样之间的差距。

## 27. 生成映射仍然受到随机配对学习方式的影响

Shortcut 的基础速度场来自：

$$
v(x,t)
=
\mathbb E[x_1-x_0\mid x_t=x].
$$

因此，生成映射仍然依赖于数据分布及所选择的噪声—数据耦合方式。

论文指出，如何进一步改善这种映射、降低一步与多步之间的差距，是后续值得研究的问题。

---

# 第八部分：数学补充——Shortcut 为什么能学习 ODE 流映射？

这一部分是对论文核心公式的进一步数学解释，帮助把自一致性与真实 ODE 联系起来。

## 28. Shortcut 本质上在学习什么？

定义 ODE 流映射：

$$
\boxed{
\Phi_{t+d\leftarrow t}(x)
}
\tag{28.1}
$$

表示从时间 $t$ 的状态 $x$ 出发，沿 ODE 积分 $d$ 时间后得到的状态。

那么精确 Shortcut 为：

$$
\boxed{
s^*(x,t,d)
=
\frac{
\Phi_{t+d\leftarrow t}(x)-x
}{d}.
}
\tag{28.2}
$$

因此：

$$
\boxed{
\Phi_{t+d\leftarrow t}(x)
=
x+d\,s^*(x,t,d).
}
\tag{28.3}
$$

Shortcut 实际上是对有限时间流映射的一种速度形式参数化。

## 29. 为什么组合关系必然成立？

ODE 流映射满足：

$$
\boxed{
\Phi_{t+2d\leftarrow t}
=
\Phi_{t+2d\leftarrow t+d}
\circ
\Phi_{t+d\leftarrow t}.
}
\tag{29.1}
$$

这意味着：

从 $t$ 直接积分到 $t+2d$，与先积分到 $t+d$、再积分到 $t+2d$，结果相同。

代入：

$$
\Phi_{t+d\leftarrow t}(x)
=
x+d\,s^*(x,t,d),
$$

就能得到：

$$
\boxed{
\begin{aligned}
s^*(x,t,2d)
={}&
\frac12s^*(x,t,d)\\
&+
\frac12s^*
\left(
x+d\,s^*(x,t,d),
t+d,d
\right).
\end{aligned}
}
\tag{29.2}
$$

所以 Shortcut 的训练公式不是凭经验设计的。

它来自精确 ODE 流映射必须满足的组合性质。

不过，训练时使用的是近似网络：

$$
s_\theta,
$$

因此自举 Target 本身可能存在误差。

这也是 EMA、weight decay 和 FM 基础监督重要的原因。

---

## 30. 自举误差如何向大步长传播？

设真实 Shortcut 为：

$$
s^*(x,t,d),
$$

网络预测：

$$
s_\theta(x,t,d)
=
s^*(x,t,d)+e_d(x,t).
$$

其中：

$$
e_d
$$

是预测误差。

构造两个半步时：

$$
\hat x_{t+d}
=
x+d\,s_\theta(x,t,d).
$$

真实中间状态：

$$
x_{t+d}^*
=
x+d\,s^*(x,t,d).
$$

因此：

$$
\boxed{
\hat x_{t+d}-x_{t+d}^*
=
d\,e_d(x,t).
}
\tag{30.1}
$$

第二个半步不仅存在速度预测误差，还会受到中间状态偏差的影响。

假设真实 Shortcut 关于空间状态满足 Lipschitz 条件：

$$
\|s^*(x,t,d)-s^*(y,t,d)\|
\leq L_d\|x-y\|.
$$

那么第二个半步由于状态偏移引入的速度误差不超过：

$$
L_d\,d\,\|e_d(x,t)\|.
$$

若两个半步的速度预测误差都被 $\varepsilon_d$ 控制，则在忽略大步长回归残差时，可以得到一个示意性的上界：

$$
\boxed{
\varepsilon_{2d}
\lesssim
\left(1+\frac{L_dd}{2}\right)\varepsilon_d.
}
\tag{30.2}
$$

如果大步长网络拟合自举 Target 还存在误差：

$$
\eta_{2d},
$$

则进一步有：

$$
\boxed{
\varepsilon_{2d}
\lesssim
\eta_{2d}
+
\left(1+\frac{L_dd}{2}\right)\varepsilon_d.
}
\tag{30.3}
$$

这里的上界是根据流映射组合关系给出的补充分析，不是论文直接给出的定理。

它说明：

**大步长误差既来自自身的回归误差，也来自两个小步长预测中的误差传播。**

因此，Shortcut 的成功依赖于较准确的小步长基础，以及稳定的自举过程。

---

# 第九部分：与 MeanFlow 的数学联系

因为我们刚刚学过 MeanFlow，这里只从数学角度补充两者之间的联系。

## 31. 两者学习的是同一个有限时间对象

Shortcut 定义：

$$
\boxed{
s(x,t,d)
=
\frac1d
\int_t^{t+d}v(x_\tau,\tau)\,d\tau.
}
\tag{31.1}
$$

MeanFlow 定义：

$$
\boxed{
u(x,t,r)
=
\frac1{r-t}
\int_t^r v(x_\tau,\tau)\,d\tau.
}
\tag{31.2}
$$

令：

$$
r=t+d,
$$

得到：

$$
\boxed{
s(x,t,d)=u(x,t,t+d).
}
\tag{31.3}
$$

所以，在精确数学定义下，它们描述的是同一个平均速度场。

区别在于训练约束。

Shortcut 使用：

$$
\boxed{
\begin{aligned}
s(x,t,2d)
={}&
\frac12s(x,t,d)\\
&+
\frac12s(x+d\,s(x,t,d),t+d,d).
\end{aligned}
}
\tag{31.4}
$$

MeanFlow 使用：

$$
\boxed{
u
=
v+(r-t)D_t^vu.
}
\tag{31.5}
$$

一个是有限区间的组合关系。

另一个是沿 ODE 轨迹的微分恒等式。

## 32. 能否从 Shortcut 推导出 MeanFlow Identity？

可以从同一个流映射关系推导。

定义：

$$
F(x,t,d)=x+d\,s(x,t,d).
$$

其中：

$$
F(x,t,d)
$$

是从时间 $t$ 出发，跨越 $d$ 的流映射。

保持目标时间：

$$
r=t+d
$$

不变。

因此：

$$
d=r-t.
$$

随着起始时间 $t$ 增加：

$$
\frac{dd}{dt}=-1.
$$

根据平均速度积分定义：

$$
d\,s(x_t,t,d)
=
\int_t^r v(x_\tau,\tau)\,d\tau.
$$

对起始时间 $t$ 求全导数：

左边：

$$
\begin{aligned}
\frac{d}{dt}[d\,s]
&=
-s+d\frac{d}{dt}s\\
&=
-s+d
\left(
J_sv+\partial_ts-\partial_ds
\right).
\end{aligned}
$$

右边：

$$
\frac{d}{dt}
\int_t^r v(x_\tau,\tau)\,d\tau
=
-v(x_t,t).
$$

因此：

$$
-s+d
\left(
J_sv+\partial_ts-\partial_ds
\right)
=
-v.
$$

整理：

$$
\boxed{
s
=
v+d
\left(
J_sv+\partial_ts-\partial_ds
\right).
}
\tag{32.1}
$$

这正是将 MeanFlow Identity 改写为网络输入：

$$
(x,t,d)
$$

后的形式。

因此，两篇论文虽然训练方式不同，但都在利用精确流映射的数学结构。

---

# 第十部分：全文总结

Shortcut Models 的理论链条可以归纳为：

$$
\boxed{
\begin{gathered}
x_0\sim\mathcal N(0,I),
\qquad
x_1\sim p_{\mathrm{data}}\\[6pt]
\Downarrow\\[6pt]
x_t=(1-t)x_0+tx_1\\[6pt]
V=x_1-x_0\\[6pt]
\Downarrow\\[6pt]
\boxed{
s_\theta(x_t,t,0)\approx
\mathbb E[V\mid x_t]
}\\[6pt]
\Downarrow\\[6pt]
\text{学习瞬时速度场}\\[6pt]
\Downarrow\\[6pt]
\boxed{
\begin{aligned}
\hat x_{t+d}
&=
x_t+d\,s_\theta(x_t,t,d)\\
s_{\mathrm{target}}
&=
\frac12\Big[
s_\theta(x_t,t,d)\\
&\qquad+
s_\theta(\hat x_{t+d},t+d,d)
\Big]
\end{aligned}
}\\[6pt]
\Downarrow\\[6pt]
\boxed{
\mathcal L_{\mathrm{SC}}
=
\left\|
s_\theta(x_t,t,2d)
-
\operatorname{sg}(s_{\mathrm{target}})
\right\|^2
}\\[6pt]
\Downarrow\\[6pt]
\text{从小步长向大步长自举}\\[6pt]
\Downarrow\\[6pt]
\boxed{
\hat x_1=x_0+s_\theta(x_0,0,1)
}
\end{gathered}
}
$$

我认为这篇论文最值得记住的是三个地方。

**第一，Shortcut 并没有要求真实 ODE 轨迹变成直线。** 它只是让模型能够直接预测沿弯曲 ODE 轨迹积分后的终点。

**第二，大步长 Target 来自两个小步长的模型预测，而不是来自原始训练样本的直线插值。** 这是避免随机配对歧义、学习边缘 ODE 流映射的关键。

**第三，Shortcut 能从零训练的基础是 $d=0$ 处的 Flow Matching 监督。** 自一致性负责把小步长的生成能力逐渐传递到大步长，FM 则负责让整个系统学习正确的数据分布。

最终可以用一句话概括：

$$
\boxed{
\text{Shortcut}
=
\text{Flow Matching 基础监督}
+
\text{二叉时间自举}
+
\text{步长条件化生成}.
}
$$

这篇论文在方法设计上的重要性，是把原本需要独立训练教师、逐阶段蒸馏的过程，转化为**同一个模型、同一次训练中的多时间尺度自举学习**。

**原始资料：** [论文全文](https://arxiv.org/html/2410.12557) · [PDF](https://arxiv.org/pdf/2410.12557) · [官方代码](https://github.com/kvfrans/shortcut-models)